# pap_17stone_resimp_dual_ML
## Sigmoid-Blended Physics-Informed Hybrid with Dual Gauss-Weighted CPP for Predicting Impact Resistance of Carbonate Natural Stones

**Objective:** Develop and validate a three-layer hybrid ML framework (SBPIH + Dual CPP) to predict impact resistance (IR, kPa) of carbonate natural stones from standard mechanical test results, without data leakage.

**Architecture:**
- Layer 1: Extra Trees regressor (RC, KH, SMI_mul, Size)
- Layer 2: IS-activated sigmoid blend (k=5) for extrapolation handling
- Layer 3: Dual CPP — Channel A: Fe₂O₃+RC (Gauss) | Channel B: LoI (Gauss)

**Validation:** LOGO-17fold CV | Bootstrap CI (n=2000) | Wilcoxon | Permutation test

**Dataset:** `res_imp_51obs_v4.xlsx` | 51 observations | 17 stone types × 3 thicknesses | Sarıışık et al. (2016, 2017)

**Target:** IR (kPa) — BS EN 14158

In [ ]:
# ============================================================
# 1. Google Drive Mount
# ============================================================
from google.colab import drive
drive.mount('/drive')

import os
BASE_DIR   = '/drive/MyDrive/Armoren/pap_res_imp'
DATA_DIR   = f'{BASE_DIR}/data'
OUTPUT_DIR = f'{BASE_DIR}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATASET = 'res_imp_51obs_v4.xlsx'
print(f'✓ Drive mounted')
print(f'  BASE_DIR  : {BASE_DIR}')
print(f'  DATA_DIR  : {DATA_DIR}')
print(f'  OUTPUT_DIR: {OUTPUT_DIR}')
print(f'  DATASET   : {DATASET}')

In [ ]:
# ============================================================
# 2. Install Dependencies
# ============================================================
!pip install -q scikit-learn openpyxl boruta shap scipy statsmodels
print('✓ All dependencies installed')

In [ ]:
# ============================================================
# 3. Imports & Global Constants
# ============================================================
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon, kruskal, spearmanr
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools import add_constant
from boruta import BorutaPy
import shap
import warnings
warnings.filterwarnings('ignore')

TARGET  = 'Ft_IR'
SEED    = 42
N_EST   = 500
K_SIG   = 5.0
PHYS_LR = ['IS', 'Size']

NAME_MAP = {1:'T1',2:'T2',3:'T3',4:'T4',5:'T5',
            6:'M1',7:'M2',8:'M3',9:'M4',10:'M5',11:'M6',12:'M7',
            13:'K1',14:'K2',15:'K3',16:'K4',17:'K5'}
LITO_MAP = {1:'Travertine',2:'Travertine',3:'Travertine',4:'Travertine',5:'Travertine',
            6:'Marble',7:'Marble',8:'Marble',9:'Marble',10:'Marble',11:'Marble',12:'Marble',
            13:'Limestone',14:'Limestone',15:'Limestone',16:'Limestone',17:'Limestone'}

np.random.seed(SEED)
print(f'✓ Imports OK  |  SEED={SEED}  N_EST={N_EST}  K_SIG={K_SIG}')

## 4. Data Loading & Feature Engineering
`res_imp_51obs_v4.xlsx` — 51 independent observations (17 types × 3 thicknesses)
SMI_mul = KHₙ × CSₙ × BSₙ — multiplicative composite index (Q-system rationale)

In [ ]:
# ============================================================
# 4. Data Loading & SMI_mul (Multiplicative — Q-system rationale)
# ============================================================
df = pd.read_excel(f'{DATA_DIR}/{DATASET}')

scaler = MinMaxScaler()
kh_n = scaler.fit_transform(df[['KH']]).flatten()
cs_n = scaler.fit_transform(df[['CS']]).flatten()
bs_n = scaler.fit_transform(df[['BS']]).flatten()
df['SMI_mul'] = kh_n * cs_n * bs_n

# FEATS is NOT defined here — it will be determined by the
# feature selection pipeline (Boruta → Forward/Backward).

y = df[TARGET].values
groups = df['Type'].values
stone_types = sorted(df['Type'].unique())

print(f'✓ Shape: {df.shape}')
print(f'  IR range: {y.min():.2f} – {y.max():.2f} kPa')
print(f'  SMI_mul:  {df["SMI_mul"].min():.4f} – {df["SMI_mul"].max():.4f}')


## 5. Data Leakage Detection
IR = Wt/V — any variable from the BS EN 14158 impact test is algebraically embedded in the target.

In [ ]:
# ============================================================
# 5. Data Leakage Detection
# ============================================================
# These variables are algebraically embedded in IR = Wt/V.
# They must never enter the model as predictors.
LEAKAGE_VARS = ['Wt', 'Height', 'RE', 'n', 'FS', 'RC2', 'v0RC', 'RE_RC2']

print('DATA LEAKAGE CHECK')
print('=' * 50)
print('  Variables excluded from all candidate sets:')
for v in LEAKAGE_VARS:
    in_data = v in df.columns
    print(f'  {v:<10} {"present in data — EXCLUDED" if in_data else "not in data"}')
print(f'\n  ✓ Leakage variables identified and excluded')


## 6. Table 1 — Descriptive Statistics

In [ ]:
# ============================================================
# 6. Table 1 — Descriptive Statistics
# ============================================================
# Descriptive table covers the four model candidates + target.
# These are the variables that enter feature selection.
DESC_COLS = ['RC', 'KH', 'SMI_mul', 'Size', TARGET]
T1 = pd.DataFrame([{
    'Variable': col, 'N': len(df[col]),
    'Mean': round(df[col].mean(), 4), 'SD': round(df[col].std(), 4),
    'Min': round(df[col].min(), 4), 'Q25': round(df[col].quantile(0.25), 4),
    'Median': round(df[col].median(), 4), 'Q75': round(df[col].quantile(0.75), 4),
    'Max': round(df[col].max(), 4), 'CV_pct': round(df[col].std() / df[col].mean() * 100, 2)
} for col in DESC_COLS])
print(T1.to_string(index=False))


## 7. Boruta Feature Selection (perc 75–100 sweep)

In [ ]:
# ============================================================
# 7. Boruta Feature Selection
# ============================================================
ALL_FEATS = ['RC', 'KH', 'CS', 'BS', 'IS', 'Density', 'Por', 'SMI_mul', 'Size']
ALL_GRP = {'RC':'Test', 'KH':'Physical', 'CS':'Physical', 'BS':'Physical',
           'IS':'Physical', 'Density':'Physical', 'Por':'Physical', 'SMI_mul':'Physical', 'Size':'Design'}

boruta_scan = []
for perc in [75, 80, 85, 90, 95, 100]:
    et_b = ExtraTreesRegressor(n_estimators=N_EST, random_state=SEED, n_jobs=-1)
    boruta = BorutaPy(et_b, n_estimators='auto', perc=perc,
                      max_iter=100, random_state=SEED, verbose=0)
    boruta.fit(df[ALL_FEATS].values, y)
    for i, col in enumerate(ALL_FEATS):
        status = ('Confirmed' if boruta.support_[i]
                  else ('Tentative' if boruta.support_weak_[i] else 'Rejected'))
        boruta_scan.append({'perc': perc, 'Feature': col, 'Ranking': boruta.ranking_[i],
                            'Status': status, 'Group': ALL_GRP[col]})

BORUTA_df = pd.DataFrame(boruta_scan)
print('BORUTA RESULTS (perc sweep)')
print('=' * 50)
for feat in ALL_FEATS:
    sub = BORUTA_df[BORUTA_df['Feature'] == feat]
    conf = sum(s == 'Confirmed' for s in sub['Status'])
    print(f'  {feat:<10} Confirmed: {conf}/6  → {sub.iloc[-1]["Status"]}')

# Chemical variables — CPP channel candidates
print(f'\n  CPP channel candidates (Boruta perc=90):')
chem = ['Fe2O3', 'LoI', 'K2O', 'MgO']
chem_avail = [c for c in chem if c in df.columns]
et_c = ExtraTreesRegressor(n_estimators=N_EST, random_state=SEED, n_jobs=-1)
boruta_c = BorutaPy(et_c, n_estimators='auto', perc=90, max_iter=100, random_state=SEED, verbose=0)
boruta_c.fit(df[ALL_FEATS + chem_avail].values, y)
for i, col in enumerate(ALL_FEATS + chem_avail):
    if col in chem_avail:
        status = ('Confirmed' if boruta_c.support_[i]
                  else ('Tentative' if boruta_c.support_weak_[i] else 'Rejected'))
        print(f'    {col:<10} → {status}  (not in main model — CPP channel)')

## 7b. Feature Selection (Forward/Backward)
Boruta confirmed features enter forward and backward selection under LOGO-17fold CV.
**FEATS is determined here — not hardcoded.**

In [ ]:
# ============================================================
# 7b. FEATURE SELECTION VALIDATION (Forward + Backward)
# This cell DETERMINES FEATS from data — no hardcoding.
# ============================================================

PERC_GRID = [75, 80, 85, 90, 95, 100]

# ── PHASE 1: Perc Sweep ─────────────────────────────────────
print('PHASE 1: BORUTA PERC SWEEP')
print('=' * 75)
print(f'  {"perc":>4} | {"Confirmed features":<45} | {"N":>2} | {"MAPE%":>7} | {"R²":>7}')
print(f'  {"-"*72}')

perc_results = []
for perc in PERC_GRID:
    conf = BORUTA_df[(BORUTA_df['perc'] == perc) & (BORUTA_df['Status'] == 'Confirmed')]
    conf_feats = list(conf['Feature'].values)
    model_feats = []
    has_static = False
    for f in conf_feats:
        if f in ['CS', 'BS']:
            has_static = True
            continue
        if f == 'SMI_mul':
            has_static = True
            model_feats.append('SMI_mul')
        else:
            model_feats.append(f)
    if has_static and 'SMI_mul' not in model_feats:
        model_feats.append('SMI_mul')
    if not model_feats:
        continue
    yp_perc = np.zeros_like(y, dtype=float)
    for t in stone_types:
        te = np.where(groups == t)[0]
        tr = np.where(groups != t)[0]
        et = ExtraTreesRegressor(n_estimators=N_EST, random_state=SEED, n_jobs=-1)
        et.fit(df[model_feats].values[tr], y[tr])
        yp_perc[te] = et.predict(df[model_feats].values[te])
    r2_perc = r2_score(y, yp_perc)
    mape_perc = np.mean(np.abs((y - yp_perc) / y)) * 100
    perc_results.append({'perc': perc, 'features': model_feats.copy(), 'n_feat': len(model_feats), 'mape': mape_perc, 'r2': r2_perc})
    feat_str = ', '.join(model_feats)
    print(f'  {perc:>4} | {feat_str:<45} | {len(model_feats):>2} | {mape_perc:>6.2f}% | {r2_perc:>6.4f}')

best_phase1 = min(perc_results, key=lambda x: x['mape'])
print(f'\n  Phase 1 best: perc={best_phase1["perc"]}, MAPE={best_phase1["mape"]:.2f}%')
print(f'  Full confirmed set: {best_phase1["features"]}')

# ── PHASE 2: Backward Elimination ───────────────────────────
print(f'\n{"="*75}')
print('PHASE 2: BACKWARD FEATURE ELIMINATION')
print('=' * 75)
current_feats = best_phase1['features'].copy()
current_mape = best_phase1['mape']
eliminated = []
iteration = 0
while True:
    iteration += 1
    print(f'  Iteration {iteration}: {current_feats} (MAPE={current_mape:.2f}%)')
    best_drop = None
    best_drop_mape = current_mape
    for feat in current_feats:
        if len(current_feats) <= 2:
            continue
        test_feats = [f for f in current_feats if f != feat]
        yp_t = np.zeros_like(y, dtype=float)
        for t in stone_types:
            te = np.where(groups == t)[0]
            tr = np.where(groups != t)[0]
            et = ExtraTreesRegressor(n_estimators=N_EST, random_state=SEED, n_jobs=-1)
            et.fit(df[test_feats].values[tr], y[tr])
            yp_t[te] = et.predict(df[test_feats].values[te])
        mape_t = np.mean(np.abs((y - yp_t) / y)) * 100
        tag = '✓ BETTER' if mape_t < current_mape else ''
        print(f'    Drop {feat:<10} → MAPE={mape_t:.2f}% {tag}')
        if mape_t < best_drop_mape:
            best_drop = feat
            best_drop_mape = mape_t
    if best_drop:
        print(f'  → Eliminating: {best_drop} (MAPE: {current_mape:.2f}% → {best_drop_mape:.2f}%)')
        current_feats = [f for f in current_feats if f != best_drop]
        current_mape = best_drop_mape
        eliminated.append(best_drop)
    else:
        print(f'  → Stopping.')
        break
bwd_feats = current_feats.copy()
bwd_mape = current_mape
print(f'  Backward result: {bwd_feats} (MAPE={bwd_mape:.2f}%)')

# ── PHASE 3: Forward Selection ──────────────────────────────
print(f'\n{"="*75}')
print('PHASE 3: FORWARD FEATURE SELECTION')
print('=' * 75)
all_candidates = best_phase1['features'].copy()
fwd_feats = []
fwd_mape = 999.0
iteration_f = 0
while True:
    iteration_f += 1
    remaining = [f for f in all_candidates if f not in fwd_feats]
    if not remaining:
        break
    print(f'  Iteration {iteration_f}: {fwd_feats if fwd_feats else "(empty)"} (MAPE={fwd_mape:.2f}%)')
    best_add = None
    best_add_mape = fwd_mape
    for feat in remaining:
        test_feats = fwd_feats + [feat]
        yp_t = np.zeros_like(y, dtype=float)
        for t in stone_types:
            te = np.where(groups == t)[0]
            tr = np.where(groups != t)[0]
            et = ExtraTreesRegressor(n_estimators=N_EST, random_state=SEED, n_jobs=-1)
            et.fit(df[test_feats].values[tr], y[tr])
            yp_t[te] = et.predict(df[test_feats].values[te])
        mape_t = np.mean(np.abs((y - yp_t) / y)) * 100
        tag = '✓ BETTER' if mape_t < fwd_mape else ''
        print(f'    Add {feat:<10} → MAPE={mape_t:.2f}% {tag}')
        if mape_t < best_add_mape:
            best_add = feat
            best_add_mape = mape_t
    if best_add:
        print(f'  → Adding: {best_add} (MAPE: {fwd_mape:.2f}% → {best_add_mape:.2f}%)')
        fwd_feats.append(best_add)
        fwd_mape = best_add_mape
    else:
        print(f'  → Stopping.')
        break
print(f'  Forward result: {fwd_feats} (MAPE={fwd_mape:.2f}%)')

# ── FEATS = Forward result (data-driven) ────────────────────
print(f'\n{"="*75}')
print('FEATURE SELECTION RESULT')
print('=' * 75)
print(f'  Backward result : {sorted(bwd_feats)} (MAPE={bwd_mape:.2f}%)')
print(f'  Forward result  : {sorted(fwd_feats)} (MAPE={fwd_mape:.2f}%)')

# Forward has lower MAPE → use forward result
FEATS = fwd_feats
print(f'\n  ✓ FEATS = {FEATS} (determined by forward selection, order preserved)')
print(f'\n✓ Feature selection complete')


## 8. Table 3 — VIF Analysis

In [ ]:
# ============================================================
# 8. VIF Analysis
# ============================================================
X_vif = add_constant(df[FEATS].values)
VIF_df = pd.DataFrame([
    {'Feature': col,
     'VIF': round(variance_inflation_factor(X_vif, i+1), 2),
     'Group': 'Test' if col == 'RC' else ('Physical' if col in ['KH','SMI_mul'] else 'Design')}
    for i, col in enumerate(FEATS)
]).sort_values('VIF', ascending=False)
print('VIF TABLE')
print(VIF_df.to_string(index=False))

## 9. Kruskal-Wallis & Spearman Correlation
Size excluded: design variable with zero between-type variance (identical 1,2,3 in every type).

In [ ]:
# ============================================================
# 9. Kruskal-Wallis H Test & Spearman Correlation
# ============================================================
STAT_FEATS = ['RC', 'KH', 'SMI_mul']

KW_rows = []
for col in STAT_FEATS + [TARGET]:
    tg = [df[df['Type'] == t][col].values for t in stone_types]
    H, p = kruskal(*tg)
    eta2 = (H - len(stone_types) + 1) / (len(df) - len(stone_types))
    KW_rows.append({'Variable': col, 'H': round(H, 2), 'p': round(p, 4),
                    'eta2': round(max(eta2, 0), 3),
                    'Sig': '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))})
KW_df = pd.DataFrame(KW_rows).sort_values('eta2', ascending=False)
print('KRUSKAL-WALLIS (k=17, n=51)')
print(KW_df.to_string(index=False))

type_means = df.groupby('Type')[STAT_FEATS + [TARGET]].mean()
SP_rows = []
for col in STAT_FEATS:
    rho, p = spearmanr(type_means[col], type_means[TARGET])
    SP_rows.append({'Feature': col, 'rho': round(rho, 3),
                    'p': '<0.001' if p < 0.001 else f'{p:.4f}'})
SP_df = pd.DataFrame(SP_rows).sort_values('rho', key=abs, ascending=False)
print(f'\nSPEARMAN (n=17 type means)')
print(SP_df.to_string(index=False))

## 10. Sigmoid & Gauss Functions

In [ ]:
# ============================================================
# 10. Sigmoid & Gauss Membership Functions
# ============================================================
def extrap_dist(val, min_tr, max_tr, rng):
    """Normalised distance: d>0 extrapolation, d<=0 interpolation."""
    if val < min_tr:
        return (min_tr - val) / rng
    elif val > max_tr:
        return (val - max_tr) / rng
    else:
        center = (min_tr + max_tr) / 2
        return -(1.0 - abs(val - center) / (rng / 2))

def sigmoid_mu(d, k=K_SIG):
    """Sigmoid: d<=0 → μ=0 (hard floor), d>0 → μ rises."""
    return 1.0 / (1.0 + np.exp(-k * d))

def gauss_mu(x, mu, sigma):
    """Inverse Gaussian: near mean → μ≈0, far from mean → μ≈1."""
    if sigma > 0:
        return 1.0 - np.exp(-0.5 * ((x - mu) / sigma) ** 2)
    return 0.0

print('✓ Sigmoid: μ = 1/(1+exp(−k·d)),  d≤0 → μ=0')
print('✓ Gauss:   μ = 1 − exp(−0.5×((x−μ)/σ)²),  normal→0, anomaly→1')

## 11. LOGO-CV: 4-Model Layered Comparison
| Model | Architecture |
|-------|-------------|
| M1 | Pure ET (baseline) |
| M2 | ET + IS Sigmoid |
| M3 | ET + IS Sigmoid + CPP(Fe₂O₃, RC) |
| M4 | ET + IS Sigmoid + Dual CPP(Fe₂O₃+RC, LoI) |

In [ ]:
# ============================================================
# 8. LOGO-CV: 4 MODEL KARŞILAŞTIRMA
# ============================================================
# Model 1 : Saf ET (RC, KH, SMI_mul, Size) — baseline
# Model 2 : ET + IS Sigmoid — extrapolasyon düzeltme
# Model 3 : ET + IS Sigmoid + CPP(Fe₂O₃, RC) — tek kanal
# Model 4 : ET + IS Sigmoid + Dual CPP(Fe₂O₃+RC, LoI) — son model

# ── FONKSİYON: base model (ET ± sigmoid) ──────────────────────
def run_base(use_sigmoid=True):
    yp_out = np.zeros_like(y, dtype=float)
    mu_out = np.zeros_like(y, dtype=float)
    for t in stone_types:
        te = np.where(groups == t)[0]
        tr = np.where(groups != t)[0]
        et = ExtraTreesRegressor(n_estimators=N_EST, random_state=SEED, n_jobs=-1)
        et.fit(df[FEATS].values[tr], y[tr])
        yp_et = et.predict(df[FEATS].values[te])
        if use_sigmoid:
            is_tr = df['IS'].values[tr]
            is_te = df['IS'].values[te].mean()
            rng = is_tr.max() - is_tr.min()
            d_is = extrap_dist(is_te, is_tr.min(), is_tr.max(), rng)
            mu = sigmoid_mu(d_is) if d_is > 0 else 0.0
            mu_out[te] = mu
            lr = LinearRegression()
            lr.fit(df[PHYS_LR].values[tr], y[tr])
            yp_lr = lr.predict(df[PHYS_LR].values[te])
            yp_out[te] = (1 - mu) * yp_et + mu * yp_lr
        else:
            yp_out[te] = yp_et
    return yp_out, mu_out

# ── FONKSİYON: CPP katmanı ────────────────────────────────────
def run_cpp(yp_base_in, mu_IS_in, cpp_channels):
    """
    cpp_channels: list of dicts
      {'vars': ['Fe2O3','RC'], 'gauss_var': 'Fe2O3'}
      {'vars': ['LoI'], 'gauss_var': 'LoI'}
    """
    yp_out = yp_base_in.copy()
    mu_store = {}
    for ch in cpp_channels:
        gvar = ch['gauss_var']
        mu_store[gvar] = np.zeros_like(y, dtype=float)

    for t_test in stone_types:
        te = np.where(groups == t_test)[0]
        tr = np.where(groups != t_test)[0]
        tr_types = [tt for tt in stone_types if tt != t_test]

        # Gauss kapıları hesapla
        mu_vals = {}
        for ch in cpp_channels:
            gvar = ch['gauss_var']
            g_tr = df[gvar].values[tr]
            g_te = df[gvar].values[te].mean()
            mu_g = gauss_mu(g_te, g_tr.mean(), g_tr.std())
            mu_vals[gvar] = mu_g
            mu_store[gvar][te] = mu_g

        # Inner LOGO-CV residual toplama
        inner_res = []
        inner_data = {i: [] for i in range(len(cpp_channels))}

        for t_val in tr_types:
            val_idx = np.where(groups == t_val)[0]
            inn_tr = np.array([j for j in tr if groups[j] != t_val])

            is_inn = df['IS'].values[inn_tr]
            is_val = df['IS'].values[val_idx].mean()
            rng_i = is_inn.max() - is_inn.min()
            d_i = extrap_dist(is_val, is_inn.min(), is_inn.max(), rng_i)
            mu_i = sigmoid_mu(d_i) if d_i > 0 else 0.0

            et_i = ExtraTreesRegressor(n_estimators=N_EST, random_state=SEED, n_jobs=-1)
            et_i.fit(df[FEATS].values[inn_tr], y[inn_tr])
            yp_et_i = et_i.predict(df[FEATS].values[val_idx])

            lr_i = LinearRegression()
            lr_i.fit(df[PHYS_LR].values[inn_tr], y[inn_tr])
            yp_lr_i = lr_i.predict(df[PHYS_LR].values[val_idx])

            yp_val = (1 - mu_i) * yp_et_i + mu_i * yp_lr_i
            res_val = y[val_idx] - yp_val

            for ix, rv in zip(val_idx, res_val):
                inner_res.append(rv)
                for ci, ch in enumerate(cpp_channels):
                    inner_data[ci].append(df[ch['vars']].values[ix])

        # Her kanal için düzeltme
        total_corr = np.zeros(len(te))
        for ci, ch in enumerate(cpp_channels):
            gvar = ch['gauss_var']
            cvars = ch['vars']
            X_inner = np.array(inner_data[ci])
            if X_inner.ndim == 1:
                X_inner = X_inner.reshape(-1, 1)
            cl = LinearRegression()
            cl.fit(X_inner, np.array(inner_res))
            X_te = df[cvars].values[te]
            if X_te.ndim == 1:
                X_te = X_te.reshape(-1, 1)
            raw_corr = cl.predict(X_te)
            total_corr += mu_vals[gvar] * raw_corr

        yp_out[te] = yp_base_in[te] + total_corr

    return yp_out, mu_store

# ══════════════════════════════════════════════════════════════
# MODEL 1: Saf ET
# ══════════════════════════════════════════════════════════════
yp_m1, _ = run_base(use_sigmoid=False)
r2_m1 = r2_score(y, yp_m1)
mape_m1 = np.mean(np.abs((y - yp_m1) / y)) * 100
mae_m1 = mean_absolute_error(y, yp_m1)
rmse_m1 = np.sqrt(np.mean((y - yp_m1)**2))
print(f'Model 1 — Saf ET           : R²={r2_m1:.4f} | MAPE={mape_m1:.2f}% | MAE={mae_m1:.3f} | RMSE={rmse_m1:.3f}')

# ══════════════════════════════════════════════════════════════
# MODEL 2: ET + IS Sigmoid
# ══════════════════════════════════════════════════════════════
yp_m2, mu_IS = run_base(use_sigmoid=True)
r2_m2 = r2_score(y, yp_m2)
mape_m2 = np.mean(np.abs((y - yp_m2) / y)) * 100
mae_m2 = mean_absolute_error(y, yp_m2)
rmse_m2 = np.sqrt(np.mean((y - yp_m2)**2))
print(f'Model 2 — ET + Sigmoid     : R²={r2_m2:.4f} | MAPE={mape_m2:.2f}% | MAE={mae_m2:.3f} | RMSE={rmse_m2:.3f}')

# ══════════════════════════════════════════════════════════════
# MODEL 3: ET + IS Sigmoid + CPP(Fe₂O₃, RC)
# ══════════════════════════════════════════════════════════════
cpp_single = [{'vars': ['Fe2O3', 'RC'], 'gauss_var': 'Fe2O3'}]
yp_m3, mu_m3 = run_cpp(yp_m2, mu_IS, cpp_single)
r2_m3 = r2_score(y, yp_m3)
mape_m3 = np.mean(np.abs((y - yp_m3) / y)) * 100
mae_m3 = mean_absolute_error(y, yp_m3)
rmse_m3 = np.sqrt(np.mean((y - yp_m3)**2))
mu_Fe = mu_m3['Fe2O3']
print(f'Model 3 — + CPP(Fe₂O₃,RC) : R²={r2_m3:.4f} | MAPE={mape_m3:.2f}% | MAE={mae_m3:.3f} | RMSE={rmse_m3:.3f}')

# ══════════════════════════════════════════════════════════════
# MODEL 4: ET + IS Sigmoid + Dual CPP(Fe₂O₃+RC, LoI)
# ══════════════════════════════════════════════════════════════
cpp_dual = [
    {'vars': ['Fe2O3', 'RC'], 'gauss_var': 'Fe2O3'},
    {'vars': ['LoI'],         'gauss_var': 'LoI'}
]
yp_m4, mu_m4 = run_cpp(yp_m2, mu_IS, cpp_dual)
r2_m4 = r2_score(y, yp_m4)
mape_m4 = np.mean(np.abs((y - yp_m4) / y)) * 100
mae_m4 = mean_absolute_error(y, yp_m4)
rmse_m4 = np.sqrt(np.mean((y - yp_m4)**2))
mu_Fe = mu_m4['Fe2O3']
mu_LoI = mu_m4['LoI']
print(f'Model 4 — + Dual CPP      : R²={r2_m4:.4f} | MAPE={mape_m4:.2f}% | MAE={mae_m4:.3f} | RMSE={rmse_m4:.3f}')

# ══════════════════════════════════════════════════════════════
# KARŞILAŞTIRMA TABLOSU
# ══════════════════════════════════════════════════════════════
yp = yp_m4.copy()
residuals = y - yp
r2 = r2_m4; mae = mae_m4; mape = mape_m4; rmse = rmse_m4
mu_final = np.maximum(mu_IS, np.maximum(mu_Fe, mu_LoI))

print(f'\n{"="*65}')
print(f'  4 MODEL KARŞILAŞTIRMA')
print(f'{"="*65}')
print(f'  {"Model":<35} {"R²":>7} {"MAPE%":>7} {"MAE":>7} {"RMSE":>7}')
print(f'  {"-"*60}')
print(f'  {"Saf ET":<35} {r2_m1:7.4f} {mape_m1:7.2f} {mae_m1:7.3f} {rmse_m1:7.3f}')
print(f'  {"ET + IS Sigmoid":<35} {r2_m2:7.4f} {mape_m2:7.2f} {mae_m2:7.3f} {rmse_m2:7.3f}')
print(f'  {"ET + Sigmoid + CPP(Fe₂O₃,RC)":<35} {r2_m3:7.4f} {mape_m3:7.2f} {mae_m3:7.3f} {rmse_m3:7.3f}')
print(f'  {"ET + Sigmoid + Dual CPP":<35} {r2_m4:7.4f} {mape_m4:7.2f} {mae_m4:7.3f} {rmse_m4:7.3f}')

# Taş bazlı M4 karşılaştırma
print(f'\n  K5 ve M4 katman katkısı:')
for t, name in [(17,'K5'), (9,'M4'), (6,'M1'), (8,'M3')]:
    i = groups == t
    m1 = np.mean(np.abs((y[i]-yp_m1[i])/y[i]))*100
    m2 = np.mean(np.abs((y[i]-yp_m2[i])/y[i]))*100
    m3 = np.mean(np.abs((y[i]-yp_m3[i])/y[i]))*100
    m4 = np.mean(np.abs((y[i]-yp_m4[i])/y[i]))*100
    print(f'  {name}: ET={m1:.1f}% → Sig={m2:.1f}% → CPP={m3:.1f}% → Dual={m4:.1f}%')

## 12. Bootstrap CI (n=2000, stratified)

In [ ]:
# ============================================================
# 12. Bootstrap 95% Confidence Interval
# ============================================================
np.random.seed(SEED)
boot_r2 = []
for _ in range(2000):
    idx = np.concatenate([
        np.random.choice(np.where(groups == t)[0],
                         size=len(np.where(groups == t)[0]), replace=True)
        for t in stone_types])
    boot_r2.append(r2_score(y[idx], yp[idx]))

ci_low  = np.percentile(boot_r2, 2.5)
ci_high = np.percentile(boot_r2, 97.5)
mean_r2 = np.mean(boot_r2)
print(f'✓ Bootstrap: Mean R²={mean_r2:.4f}, 95% CI=[{ci_low:.4f}, {ci_high:.4f}]')

## 13. Wilcoxon Signed-Rank & Permutation Test

In [ ]:
# ============================================================
# 13. Wilcoxon Signed-Rank & Permutation Test
# ============================================================
stat_w, p_w = wilcoxon(residuals)
bias_flag = p_w < 0.05
print(f'WILCOXON: W={stat_w:.0f}, p={p_w:.4f} → {"BIAS" if bias_flag else "NO BIAS ✓"}')
print(f'  Mean Bias = {residuals.mean():.4f} kPa')

np.random.seed(SEED)
n_perm = 200
perm_r2 = [r2_score(np.random.permutation(y), yp) for _ in range(n_perm)]
perm_p = np.mean([pr >= r2 for pr in perm_r2])
print(f'PERMUTATION (n={n_perm}): p={perm_p:.4f} → {"✓" if perm_p < 0.05 else "⚠️"}')

## 14. SHAP Feature Importance

In [ ]:
# ============================================================
# 14. SHAP Feature Importance (full-data ET)
# ============================================================
et_full = ExtraTreesRegressor(n_estimators=N_EST, random_state=SEED, n_jobs=-1)
et_full.fit(df[FEATS].values, y)
explainer = shap.TreeExplainer(et_full)
shap_vals = explainer.shap_values(df[FEATS].values)

shap_mean = np.abs(shap_vals).mean(axis=0)
shap_pct  = shap_mean / shap_mean.sum() * 100
SHAP_df = pd.DataFrame([
    {'Feature': col, 'Mean_SHAP': round(shap_mean[i], 4),
     'Pct': round(shap_pct[i], 1),
     'Group': 'Test' if col == 'RC' else ('Physical' if col in ['KH','SMI_mul'] else 'Design')}
    for i, col in enumerate(FEATS)
]).sort_values('Mean_SHAP', ascending=False)
print('SHAP FEATURE IMPORTANCE')
print(SHAP_df.to_string(index=False))
test_pct = SHAP_df[SHAP_df['Group']=='Test']['Pct'].sum()
phys_pct = SHAP_df[SHAP_df['Group']=='Physical']['Pct'].sum()
design_pct = SHAP_df[SHAP_df['Group']=='Design']['Pct'].sum()
print(f'  Test: {test_pct:.1f}% | Physical: {phys_pct:.1f}% | Design: {design_pct:.1f}%')

## 15. Per-Type & Lithology Error Analysis (4 Models)

In [ ]:
# ============================================================
# 13. TAŞ BAZLI 4 MODEL KARŞILAŞTIRMA + LİTOLOJİ
# ============================================================
# Her katmanın hangi taşta ne katkı sağladığını gösterir

print('4 MODEL — TAŞ BAZLI KARŞILAŞTIRMA')
print('=' * 90)
print(f'  {"Taş":<5} {"Lito":<12} {"ET":>7} {"Sig":>7} {"CPP":>7} {"Dual":>7} {"μ_IS":>6} {"μ_Fe":>6} {"μ_LoI":>6}')
print(f'  {"-" * 75}')

T8_4model_rows = []
for t in stone_types:
    i = groups == t
    m1 = np.mean(np.abs((y[i] - yp_m1[i]) / y[i])) * 100
    m2 = np.mean(np.abs((y[i] - yp_m2[i]) / y[i])) * 100
    m3 = np.mean(np.abs((y[i] - yp_m3[i]) / y[i])) * 100
    m4 = np.mean(np.abs((y[i] - yp_m4[i]) / y[i])) * 100
    row = {
        'Type': t, 'Stone': NAME_MAP[t], 'Lithology': LITO_MAP[t],
        'IR_mean': round(y[i].mean(), 3),
        'Pred_ET': round(yp_m1[i].mean(), 3),
        'Pred_Sig': round(yp_m2[i].mean(), 3),
        'Pred_CPP': round(yp_m3[i].mean(), 3),
        'Pred_Dual': round(yp_m4[i].mean(), 3),
        'MAPE_ET': round(m1, 2),
        'MAPE_Sig': round(m2, 2),
        'MAPE_CPP': round(m3, 2),
        'MAPE_Dual': round(m4, 2),
        'mu_IS': round(mu_IS[i].mean(), 3),
        'mu_Fe': round(mu_Fe[i].mean(), 3),
        'mu_LoI': round(mu_LoI[i].mean(), 3),
        'mu_final': round(mu_final[i].mean(), 3),
        'R2': round(r2_score(y[i], yp_m4[i]), 3) if len(set(y[i])) > 1 else float('nan'),
        'MAE': round(mean_absolute_error(y[i], yp_m4[i]), 3),
        'Bias': round((yp_m4[i] - y[i]).mean(), 3)
    }
    T8_4model_rows.append(row)
    # En iyi katmanı işaretle
    best = min(m1, m2, m3, m4)
    flag_et   = ' ◄' if m1 == best else ''
    flag_sig  = ' ◄' if m2 == best else ''
    flag_cpp  = ' ◄' if m3 == best else ''
    flag_dual = ' ◄' if m4 == best else ''
    print(f'  {NAME_MAP[t]:<5} {LITO_MAP[t]:<12} {m1:6.1f}{flag_et} {m2:6.1f}{flag_sig} {m3:6.1f}{flag_cpp} {m4:6.1f}{flag_dual} {mu_IS[i].mean():6.3f} {mu_Fe[i].mean():6.3f} {mu_LoI[i].mean():6.3f}')

T8_4model = pd.DataFrame(T8_4model_rows)

# Litoloji bazlı 4 model
print(f'\n  LİTOLOJİ BAZLI')
print(f'  {"Lito":<12} {"ET":>7} {"Sig":>7} {"CPP":>7} {"Dual":>7}')
print(f'  {"-" * 42}')

T8b_rows = []
for lito in ['Travertine', 'Marble', 'Limestone']:
    idx = np.array([LITO_MAP[g] == lito for g in groups])
    yl = y[idx]
    m1 = np.mean(np.abs((yl - yp_m1[idx]) / yl)) * 100
    m2 = np.mean(np.abs((yl - yp_m2[idx]) / yl)) * 100
    m3 = np.mean(np.abs((yl - yp_m3[idx]) / yl)) * 100
    m4 = np.mean(np.abs((yl - yp_m4[idx]) / yl)) * 100
    r2l = r2_score(yl, yp_m4[idx])
    row = {
        'Lithology': lito,
        'MAPE_ET': round(m1, 2), 'MAPE_Sig': round(m2, 2),
        'MAPE_CPP': round(m3, 2), 'MAPE_Dual': round(m4, 2),
        'R2': round(r2l, 3),
        'MAE': round(mean_absolute_error(yl, yp_m4[idx]), 3),
        'RMSE': round(np.sqrt(np.mean((yl - yp_m4[idx])**2)), 3)
    }
    T8b_rows.append(row)
    print(f'  {lito:<12} {m1:6.2f}  {m2:6.2f}  {m3:6.2f}  {m4:6.2f}')

T8b = pd.DataFrame(T8b_rows)

# Genel 4 model
print(f'\n  GENEL')
print(f'  {"Model":<35} {"R²":>7} {"MAPE":>7}')
print(f'  {"-" * 50}')
print(f'  {"Saf ET":<35} {r2_m1:7.4f} {mape_m1:7.2f}')
print(f'  {"ET + IS Sigmoid":<35} {r2_m2:7.4f} {mape_m2:7.2f}')
print(f'  {"ET + Sigmoid + CPP(Fe₂O₃,RC)":<35} {r2_m3:7.4f} {mape_m3:7.2f}')
print(f'  {"ET + Sigmoid + Dual CPP":<35} {r2_m4:7.4f} {mape_m4:7.2f}')

## 16. Export Tables (Excel)

In [ ]:
# ============================================================
# 14. TABLOLAR → EXCEL (4 model dahil)
# ============================================================
T1 = pd.DataFrame([{
    'Variable': col, 'N': len(df[col]),
    'Mean': round(df[col].mean(), 4), 'SD': round(df[col].std(), 4),
    'Min': round(df[col].min(), 4), 'Q25': round(df[col].quantile(0.25), 4),
    'Median': round(df[col].median(), 4), 'Q75': round(df[col].quantile(0.75), 4),
    'Max': round(df[col].max(), 4), 'CV_pct': round(df[col].std() / df[col].mean() * 100, 2)
} for col in FEATS + [TARGET]])

# T5: 4 model performans
T5 = pd.DataFrame([
    {'Model': 'Saf ET', 'n_features': 4, 'CV': 'LOGO-17fold',
     'R2': round(r2_m1, 4), 'MAE_kPa': round(mae_m1, 3), 'MAPE_pct': round(mape_m1, 2), 'RMSE_kPa': round(rmse_m1, 3)},
    {'Model': 'ET + IS Sigmoid', 'n_features': 4, 'CV': 'LOGO-17fold',
     'R2': round(r2_m2, 4), 'MAE_kPa': round(mae_m2, 3), 'MAPE_pct': round(mape_m2, 2), 'RMSE_kPa': round(rmse_m2, 3)},
    {'Model': 'ET + Sigmoid + CPP(Fe₂O₃,RC)', 'n_features': 4, 'CV': 'LOGO-17fold',
     'R2': round(r2_m3, 4), 'MAE_kPa': round(mae_m3, 3), 'MAPE_pct': round(mape_m3, 2), 'RMSE_kPa': round(rmse_m3, 3)},
    {'Model': 'ET + Sigmoid + Dual CPP', 'n_features': 4, 'CV': 'LOGO-17fold',
     'R2': round(r2_m4, 4), 'MAE_kPa': round(mae_m4, 3), 'MAPE_pct': round(mape_m4, 2), 'RMSE_kPa': round(rmse_m4, 3)},
])

T6 = pd.DataFrame([{
    'Model': 'SBPIH+DualCPP',
    'Boot_Mean_R2': round(mean_r2, 4),
    'CI_95': f'[{ci_low:.4f}, {ci_high:.4f}]',
    'Mean_Bias': round(residuals.mean(), 4),
    'Wilcoxon_W': round(stat_w, 0),
    'Wilcoxon_p': round(p_w, 4),
    'Bias': 'YES' if bias_flag else 'NO',
    'Perm_p': round(perm_p, 4)
}])

with pd.ExcelWriter(f'{OUTPUT_DIR}/res_imp_tables_v5.xlsx') as writer:
    T1.to_excel(writer, sheet_name='T1_Descriptive', index=False)
    KW_df.to_excel(writer, sheet_name='T2_KruskalWallis', index=False)
    SP_df.to_excel(writer, sheet_name='T3_Spearman', index=False)
    T5.to_excel(writer, sheet_name='T5_ModelPerf', index=False)
    T6.to_excel(writer, sheet_name='T6_Bootstrap_Wilcoxon', index=False)
    SHAP_df.to_excel(writer, sheet_name='T7_SHAP', index=False)
    T8_4model.to_excel(writer, sheet_name='T8a_4Model', index=False)
    T8b.to_excel(writer, sheet_name='T8b_LithoError', index=False)
    VIF_df.to_excel(writer, sheet_name='VIF', index=False)
    BORUTA_df.to_excel(writer, sheet_name='Boruta', index=False)

print(f'✓ {OUTPUT_DIR}/res_imp_tables_v5.xlsx')

## 17. Export Figure Data (Excel)
Figures are generated separately. This cell saves raw data for all plots.

In [ ]:
# ============================================================
# 15. ŞEKİL HAM VERİLERİ → EXCEL (4 model dahil)
# ============================================================
fig_pred = pd.DataFrame({
    'Type': groups, 'Stone': [NAME_MAP[t] for t in groups],
    'Lithology': [LITO_MAP[t] for t in groups],
    'Actual_IR': y,
    'Pred_ET': yp_m1, 'Pred_Sig': yp_m2, 'Pred_CPP': yp_m3, 'Pred_Dual': yp_m4,
    'Residual': residuals, 'mu_final': mu_final
})

d_range = np.linspace(-1.5, 1.5, 300)
fig_sig = pd.DataFrame({
    'd': d_range,
    'mu_k1': [sigmoid_mu(d, k=1) if d > 0 else 0.0 for d in d_range],
    'mu_k5': [sigmoid_mu(d, k=5) if d > 0 else 0.0 for d in d_range],
    'mu_k10': [sigmoid_mu(d, k=10) if d > 0 else 0.0 for d in d_range],
})

fig_boot = pd.DataFrame({
    'R2_boot': boot_r2, 'Observed_R2': [r2] * len(boot_r2),
    'CI_low': [ci_low] * len(boot_r2), 'CI_high': [ci_high] * len(boot_r2)
})

fig_shap = pd.DataFrame(shap_vals, columns=[f'SHAP_{f}' for f in FEATS])
for f in FEATS:
    fig_shap[f'Val_{f}'] = df[f].values
fig_shap['Stone'] = [NAME_MAP[t] for t in groups]
fig_shap['Lithology'] = [LITO_MAP[t] for t in groups]

fig_framework = pd.DataFrame({
    'Type': groups, 'Stone': [NAME_MAP[t] for t in groups],
    'Lithology': [LITO_MAP[t] for t in groups],
    'IS': df['IS'].values, 'Fe2O3': df['Fe2O3'].values, 'LoI': df['LoI'].values,
    'mu_IS': mu_IS, 'mu_Fe': mu_Fe, 'mu_LoI': mu_LoI, 'mu_final': mu_final,
    'y_actual': y, 'y_pred': yp, 'residual': residuals,
    'regime': ['Extrapolation' if mu_final[i] > 0.5 else 'Interpolation' for i in range(len(y))]
})

STAT_FEATS_CORR = ['RC', 'KH', 'SMI_mul', 'Ft_IR']
type_means_corr = df.groupby('Type')[STAT_FEATS_CORR].mean()
fig_spearman = type_means_corr.corr(method='spearman').round(3)

with pd.ExcelWriter(f'{OUTPUT_DIR}/res_imp_figdata_v5.xlsx') as writer:
    fig_pred.to_excel(writer, sheet_name='Fig_PredVsActual', index=False)
    fig_sig.to_excel(writer, sheet_name='Fig_Sigmoid_curve', index=False)
    T8_4model.to_excel(writer, sheet_name='Fig_TypeError', index=False)
    fig_boot.to_excel(writer, sheet_name='Fig_Bootstrap', index=False)
    fig_shap.to_excel(writer, sheet_name='Fig_SHAP_beeswarm', index=False)
    SHAP_df.to_excel(writer, sheet_name='Fig_SHAP_bar', index=False)
    BORUTA_df.to_excel(writer, sheet_name='Fig_Boruta', index=False)
    fig_framework.to_excel(writer, sheet_name='Fig_Framework', index=False)
    fig_spearman.to_excel(writer, sheet_name='Fig_Spearman', index=True)

print(f'✓ {OUTPUT_DIR}/res_imp_figdata_v5.xlsx')

## 18. Summary

In [ ]:
# ============================================================
# 18. Summary
# ============================================================
print('=' * 65)
print('  SBPIH + Dual CPP (Fe₂O₃ + LoI) — FINAL MODEL')
print('=' * 65)
print(f'  Features : {FEATS}')
print(f'  Sigmoid  : IS (k={K_SIG})')
print(f'  CPP-A    : Fe₂O₃ + RC (Gauss)')
print(f'  CPP-B    : LoI (Gauss)')
print(f'  R²       = {r2:.4f}')
print(f'  MAE      = {mae:.3f} kPa')
print(f'  MAPE     = {mape:.2f}%')
print(f'  RMSE     = {rmse:.3f} kPa')
print(f'  Boot. CI = [{ci_low:.4f}, {ci_high:.4f}]')
print(f'  Wilcoxon = p={p_w:.4f} → {"BIAS" if bias_flag else "NO BIAS"}')
print(f'  Perm.    = p={perm_p:.4f}')
print()
print(f'  4 MODEL COMPARISON:')
print(f'  {"Model":<35} {"R²":>7} {"MAPE":>7}')
print(f'  {"-"*50}')
print(f'  {"Pure ET":<35} {r2_m1:7.4f} {mape_m1:7.2f}')
print(f'  {"ET + IS Sigmoid":<35} {r2_m2:7.4f} {mape_m2:7.2f}')
print(f'  {"+ CPP(Fe₂O₃, RC)":<35} {r2_m3:7.4f} {mape_m3:7.2f}')
print(f'  {"+ Dual CPP(Fe₂O₃+RC, LoI)":<35} {r2_m4:7.4f} {mape_m4:7.2f}')
print('=' * 65)